### BaseLine

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = pd.read_csv("../data/documents.csv")
test = pd.read_csv("../data/test_queries.csv")

vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
doc_mat = vec.fit_transform(docs["title"] + ". " + docs["text"])
sims = cosine_similarity(vec.transform(test["query"]), doc_mat)

rows = []
for i, qid in enumerate(test["query_id"]):
    for rank in sims[i].argsort()[::-1][:5]:          # best-first order
        rows.append({"QueryId": str(qid),
                     "DocumentId": str(docs.iloc[rank]["document_id"])})

pd.DataFrame(rows).to_csv("../submission2.csv", index=False)

In [2]:
docs.columns

Index(['document_id', 'title', 'text', 'source', 'crop', 'country', 'origin',
       'source_url', 'license'],
      dtype='str')

In [3]:
!pip install -q IProgress


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\HomePC\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


### LandChain CSV Data Handler

In [4]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="../data/documents.csv", encoding="utf-8",
                       csv_args={
        "delimiter": ",",
        "quotechar": '"',
        "fieldnames": ['document_id', 'title', 'text', 'source', 'crop', 'country', 'origin',
       'source_url', 'license'],
    })

C:\Users\HomePC\AppData\Local\Temp\ipykernel_8972\893629648.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
c:\Users\HomePC\Desktop\Agric_extension\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
csv_loader = loader.load()

In [6]:
csv_loader[0].page_content[0:1000]

'document_id: document_id\ntitle: title\ntext: text\nsource: source\ncrop: crop\ncountry: country\norigin: origin\nsource_url: source_url\nlicense: license'

In [7]:
csv_loader[0]

Document(metadata={'source': '../data/documents.csv', 'row': 0}, page_content='document_id: document_id\ntitle: title\ntext: text\nsource: source\ncrop: crop\ncountry: country\norigin: origin\nsource_url: source_url\nlicense: license')

In [8]:
csv_loader[0].page_content.split("\n")

['document_id: document_id',
 'title: title',
 'text: text',
 'source: source',
 'crop: crop',
 'country: country',
 'origin: origin',
 'source_url: source_url',
 'license: license']

In [9]:
def process_csv(csv_directory):
    """
    Process all CSV files in the specified directory and return a list of documents.

    Args:
        csv_directory (str): The path to the directory containing CSV files.
        """
    all_documents = []
    csv_dir = Path(csv_directory)

    csv_files = list(csv_dir.glob("**/*.csv"))

    print(f"Found {len(csv_files)} CSV files in {csv_directory}")

    for csv_file in csv_files:
        print(f"Processing {csv_file}...")
        try:
            loader = CSVLoader(str(csv_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = str(csv_file.name)
                doc.metadata["file_type"] = "csv"

            all_documents.extend(documents)
            print(f"Successfully processed {len(documents)} documents from {csv_file.name}.")
        except Exception as e:
            print(f"Error occurred while processing {csv_file}: {e}")

        print(f"Total documents processed: {len(all_documents)}\n")
    return all_documents

# all_csv_documents = process_csv("../data")


### Data Chunking

In [10]:
# import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
### Text Splitting Get Into Chunking

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks.

    Args:
        documents (list): List of Document objects to be split.
        chunk_size (int): The maximum size of each chunk.
        chunk_overlap (int): The number of overlapping characters between chunks.

    Returns:
        list: A list of Document objects representing the split chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")


    return split_docs

In [12]:
docs_test = loader.load_and_split(text_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20))

In [13]:
chunks = split_documents(csv_loader, chunk_size=1000, chunk_overlap=200)

Split 696 documents into 842 chunks.

Example Chunk:
Content: document_id: document_id
title: title
text: text
source: source
crop: crop
country: country
origin: origin
source_url: source_url
license: license...
Metadata: {'source': '../data/documents.csv', 'row': 0}


In [14]:
chunks[:5]

[Document(metadata={'source': '../data/documents.csv', 'row': 0}, page_content='document_id: document_id\ntitle: title\ntext: text\nsource: source\ncrop: crop\ncountry: country\norigin: origin\nsource_url: source_url\nlicense: license'),
 Document(metadata={'source': '../data/documents.csv', 'row': 1}, page_content='document_id: 1\ntitle: Drought and erratic rainfall: the risk to crops\ntext: Drought and erratic rainfall and its impact on farming. Late, short or broken rains cause poor germination, wilting at flowering, and sharp yield loss, especially on sandy low-organic soils.\nsource: FAO\ncrop: (general)\ncountry: Kenya\norigin: synthetic\nsource_url: \nlicense: synthetic (CC0)'),
 Document(metadata={'source': '../data/documents.csv', 'row': 2}, page_content='document_id: 2\ntitle: Adapting to drought and erratic rainfall (Guinea savanna)\ntext: Adapting to drought and erratic rainfall in the Guinea savanna, with one long rainy season and moderately fertile soils. Grow early-matur

In [15]:
print(chunks[35].page_content)

and G75 excelled specifically in pre-anthesis drought stress. This research highlights the importance of selecting and breeding drought-adapted sorghum genotypes to enhance resilience and ensure food security in drought-prone areas.


In [16]:
print(chunks[6].page_content)

document_id: 6
title: Flooding and excess rain: the risk to crops
text: Flooding and excess rain and its impact on farming. Heavy downpours waterlog roots, wash away seed and fertiliser, and spread water-borne disease, drowning crops on flat ground.
source: National Extension Service
crop: (general)
country: Mali
origin: synthetic
source_url: 
license: synthetic (CC0)


In [17]:
print(chunks[35].metadata)

{'source': '../data/documents.csv', 'row': 24}


In [18]:
for i, content in enumerate(chunks[:5]):
    print(f"Chunk {i+1}:")
    print(f"Content: {content.page_content[:200]}...")
    # print(f"Metadata: {content.metadata}")
    print("\n")

Chunk 1:
Content: document_id: document_id
title: title
text: text
source: source
crop: crop
country: country
origin: origin
source_url: source_url
license: license...


Chunk 2:
Content: document_id: 1
title: Drought and erratic rainfall: the risk to crops
text: Drought and erratic rainfall and its impact on farming. Late, short or broken rains cause poor germination, wilting at flowe...


Chunk 3:
Content: document_id: 2
title: Adapting to drought and erratic rainfall (Guinea savanna)
text: Adapting to drought and erratic rainfall in the Guinea savanna, with one long rainy season and moderately fertile ...


Chunk 4:
Content: document_id: 3
title: Adapting to drought and erratic rainfall (Highlands)
text: Adapting to drought and erratic rainfall in the cool highlands, where heavy dew and mild temperatures favour foliar fun...


Chunk 5:
Content: document_id: 4
title: Adapting to drought and erratic rainfall (Humid forest)
text: Adapting to drought and erratic rainfall in the humid fo

### Embedding

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class EmbeddingManager:
    """Any class that manages embeddings should inherit from this class and implement the required methods."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the EmbeddingManager with a specified model name.
        
        Args:
            model_name (str): The name of the embedding model to use. Default is "all
            -MiniLM-L6-v2
            """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")
            print("Embedding Dimension:", self.model.get_embedding_dimension())
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of texts to generate embeddings for.

        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings generated by the model.

        Returns:
            int: The dimension of the embeddings.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before getting embedding dimension.")
        
        return self.model.get_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1617.75it/s]


Loaded embedding model: all-MiniLM-L6-v2
Embedding Dimension: 384


In [21]:
## Document Structure

from langchain_core.documents import Document

#### Storing In Database

In [22]:
class VectorStore:
    """Any class that manages a vector store should inherit from this class and implement the required methods."""

    def __init__(self, collection_name: str = "csv_documents",persistence_directory: str = "../data/vector_store"):
        """Initialize the VectorStore with a collection name and an embedding manager.
        
        Args:
            collection_name (str): The name of the collection in the vector store.
            persistence_directory (str): The directory where the vector store will be persisted.
        """
        self.collection_name = collection_name
        self.persistence_directory = persistence_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the vector store client and collection."""
        try:
            # Creat persistence ChromaDB client
            self.client = chromadb.PersistentClient(path=self.persistence_directory)

            # Create or get the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "CSV document embeddings"})
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents: List[Document], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store.
        
        Args:
            documents (List[Document]): A list of Document objects to add to the vector store.
            embeddings (np.ndarray): An array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        print(f"Adding {len(documents)} documents to the vector store...")
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            print(doc_id)
            ids.append(doc_id)

            # Prepare metadata for the document
            metadata = dict(doc.metadata)  # Create a copy of the metadata dictionary
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embeddings[i])

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized with collection: csv_documents
Existing documents in the collection: 3368


In [23]:
texts = [doc.page_content for doc in chunks]

# texts
# Generate embeddings for the texts
embeddings = embedding_manager.generate_embeddings(texts)
# embedding_manager.get_embedding_dimension()
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 842 texts...
Generated embeddings with shape: (842, 384)
Adding 842 documents to the vector store...
doc_898d9238_0
doc_1686f9df_1
doc_260fcbbf_2
doc_85f40341_3
doc_ab9bd7a9_4
doc_dc34d1a7_5
doc_6ca1c923_6
doc_07126713_7
doc_36b29979_8
doc_25a16a89_9
doc_36d0a055_10
doc_76df1fbc_11
doc_c7345291_12
doc_4e2f0e6b_13
doc_c2340993_14
doc_07805b41_15
doc_5458caaf_16
doc_b6f924c3_17
doc_45124027_18
doc_d637fd8b_19
doc_c323a2b8_20
doc_e73a539c_21
doc_8f69d3bb_22
doc_4349332f_23
doc_c5c623f3_24
doc_b8dbfb5d_25
doc_3c10c338_26
doc_0c6ee6c5_27
doc_28538f57_28
doc_fa0309d3_29
doc_f2cb3bf2_30
doc_655f1113_31
doc_5d308fc4_32
doc_86a2891a_33
doc_5d468cc9_34
doc_57f2c9b3_35
doc_d4c6356b_36
doc_65efc202_37
doc_89b2f6c6_38
doc_9982683d_39
doc_715eff67_40
doc_5cb03a8a_41
doc_ea0d9d3e_42
doc_5e6fa25a_43
doc_edea632a_44
doc_57307ec8_45
doc_9e1a7eaa_46
doc_a0fac9ea_47
doc_ee0cae71_48
doc_814aeb80_49
doc_bb311cf6_50
doc_50b542c6_51
doc_2ad3900a_52
doc_a623d280_53
doc_64a6c980_54
doc

In [24]:
vectorstore.collection

Collection(name=csv_documents)

### Retriever

In [25]:
class RAGRetriever:
    """A class for retrieving relevant documents from a vector store based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the RAGRetriever with a vector store and an embedding manager.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve the top_k most relevant documents for a given query.
        
        Args:
            query (str): The query string to search for relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            threshold (float): The similarity threshold for retrieving documents.

        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their similarity scores.
        """

        print("Retrieving relevant documents for the query:", query)
        print(f"Using top_k={top_k} and threshold={threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(

                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance':distance,
                            'rank': i + 1

                        })
                        print(f"Retrieved {len(retrieved_docs)} documents (afer filtering)")

                    else:
                        print("No document found")

                    return retrieved_docs

        except Exception as e:
            print("Error during retrieval:",e)
            return []




rag_retriver = RAGRetriever(vectorstore,embedding_manager)
rag_retriver

In [26]:
result = rag_retriver.retrieve("Drought and erratic rainfall?", top_k=5, threshold=0.2)
result

Retrieving relevant documents for the query: Drought and erratic rainfall?
Using top_k=5 and threshold=0.2
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)


[{'id': 'doc_ee4c1cfd_1',
  'content': 'document_id: 1\ntitle: Drought and erratic rainfall: the risk to crops\ntext: Drought and erratic rainfall and its impact on farming. Late, short or broken rains cause poor germination, wilting at flowering, and sharp yield loss, especially on sandy low-organic soils.\nsource: FAO\ncrop: (general)\ncountry: Kenya\norigin: synthetic\nsource_url: \nlicense: synthetic (CC0)',
  'metadata': {'doc_index': 1,
   'row': 1,
   'content_length': 365,
   'source': '../data/documents.csv'},
  'similarity_score': 0.3160770535469055,
  'distance': 0.6839229464530945,
  'rank': 1}]

In [27]:
rag_retriver.retrieve("How do I cope with drought and erratic rainfall on my farm?", top_k=5, threshold=0.1)

Retrieving relevant documents for the query: How do I cope with drought and erratic rainfall on my farm?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)


[{'id': 'doc_ee4c1cfd_1',
  'content': 'document_id: 1\ntitle: Drought and erratic rainfall: the risk to crops\ntext: Drought and erratic rainfall and its impact on farming. Late, short or broken rains cause poor germination, wilting at flowering, and sharp yield loss, especially on sandy low-organic soils.\nsource: FAO\ncrop: (general)\ncountry: Kenya\norigin: synthetic\nsource_url: \nlicense: synthetic (CC0)',
  'metadata': {'content_length': 365,
   'source': '../data/documents.csv',
   'doc_index': 1,
   'row': 1},
  'similarity_score': 0.19625771045684814,
  'distance': 0.8037422895431519,
  'rank': 1}]

#### Retriever With Groq

In [39]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

## Initialize Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

In [29]:


llm = ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)


In [30]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F54A9B8290>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F54A8CAC00>, model_name='openai/gpt-oss-20b', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=1024)

In [31]:
from typing import List, Dict, Any

### Advance Retriever

In [32]:
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriver, llm)
result = adv_rag.query("How do I cope with drought and erratic rainfall on my farm?", top_k=5, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving relevant documents for the query: How do I cope with drought and erratic rainfall on my farm?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
document_id: 1
title: Drought and erratic rainfall: the risk to crops
text: Drought and erratic rainfall and its impact on farming. Late, short or broken rains cause poor germination, wilting at flowering, and sharp yield loss, especially on sandy low-organic soils.
source: FAO
crop: (general)
country: Kenya
origin: synthetic
source_url: 
license: synthetic (CC0)

Question: How do I cope with drought and erratic rainfall on my farm?

Answer:

Final Answer: **Coping with drought and erratic rainfall in Kenya**

| Strategy | Why it helps | Quick tip |
|----------|--------------|-----------|
| **Plant drought‑tolerant varieties** | They germinate and fl

In [33]:
result = adv_rag.query("What does sulphur deficiency look like in groundnut?", top_k=5, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving relevant documents for the query: What does sulphur deficiency look like in groundnut?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
document_id: 520
title: Identifying Sulphur deficiency in Groundnut
text: Sulphur deficiency in groundnut (Arachis hypogaea). How to recognise it in the field: General yellowing of the WHOLE plant that starts on the YOUNG upper leaves (unlike nitrogen, which hits old leaves first). Growth is stunted and spindly.
source: IITA
crop: Groundnut
country: Sudan
origin: synthetic
source_url: 
license: synthetic (CC0)

Question: What does sulphur deficiency look like in groundnut?

Answer:

Final Answer: Sulphur deficiency in groundnut shows as a general yellowing of the entire plant that begins on the young upper leaves, with stunted, spindly growth.

Citations:
[

In [34]:
result = adv_rag.query("What problem does waterlogging cause on the farm?", top_k=5, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("\nSummary:", result['summary'])
print("History:", result['history'][-1])

Retrieving relevant documents for the query: What problem does waterlogging cause on the farm?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
document_id: 682
title: Waterlogging: the problem
text: Waterlogging in smallholder farming. Standing water in flat or poorly drained fields starves roots of oxygen, yellows plants and promotes root rot.
source: AGRA
crop: (general)
country: Ghana
origin: synthetic
source_url: 
license: synthetic (CC0)

Question: What problem does waterlogging cause on the farm?

Answer:

Final Answer: Waterlogging starves crop roots of oxygen, turns plants yellow, and promotes root rot.

Citations:
[1] ../data/documents.csv (page unknown)

Summary: Waterlogging deprives crop roots of oxygen, causing them to turn yellow and become vulnerable to root rot. This lack of oxygen an

In [35]:
result.keys()

dict_keys(['question', 'answer', 'sources', 'summary', 'history'])

In [36]:
import re

def extract_document_id(result) -> int:
    """
    Extract the document ID from the result.

    Args:
        result (dict): The result dictionary containing the preview text.
    Returns:
        int: The extracted document ID, or None if not found.  
    """
    match = re.search(r'document_id:\s*(\d+)', result['sources'][0]['preview'])
    if match:
        return int(match.group(1))
    return None

In [37]:
extracted_doc_id = extract_document_id(result)
extracted_doc_id

682

### Submission

In [43]:
rows = []
for idx, qid in enumerate(test["query_id"]):
    query = test.iloc[idx]["query"]  # Get the actual query text
    # Retrieve documents for this query
    results = adv_rag.retriever.retrieve(query, top_k=5, threshold=0.1)
    
    if results:
        # Sort by similarity score (descending) and take top 5
        sorted_results = sorted(results, key=lambda x: x['similarity_score'], reverse=True)
        for result in sorted_results[:5]:
            doc_id = result['metadata'].get('document_id') or result['id']
            # Extract the latter number from doc_id (e.g., "doc_ee4c1cfd_1" -> "1")
            extracted_doc_id = doc_id.split('_')[-1] if '_' in str(doc_id) else doc_id
            rows.append({
                "QueryId": str(qid),
                "DocumentId": str(extracted_doc_id)
            })

# Save to CSV
submission_df = pd.DataFrame(rows)
submission_df.to_csv("../submission.csv", index=False)
print(f"Saved {len(rows)} query-document pairs to submission.csv")
print("\nPreview:") 
print(submission_df.head(10))

Retrieving relevant documents for the query: How do I cope with drought and erratic rainfall on my farm?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: How can I adapt my farming to drought and erratic rainfall?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: How does drought and erratic rainfall affect my crops?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: What is the risk of drought and erratic rainfall to farming?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (

In [44]:
rows = []
for idx, qid in enumerate(test["query_id"]):
    query = test.iloc[idx]["query"]  # Get the actual query text
    # Retrieve documents for this query
    results = adv_rag.retriever.retrieve(query, top_k=5, threshold=0.1)

    if not results:
        # No retrieved documents; record an empty answer row
        rows.append({
            "QueryId": str(qid),
            "Rank": None,
            "DocumentId": None,
            "Similarity": None,
      })
        continue

    # Sort by similarity score (descending) and take top 5
    sorted_results = sorted(results, key=lambda x: x['similarity_score'], reverse=True)[:5]
    for rnk, result in enumerate(sorted_results, start=1):
        doc_id = result['metadata'].get('document_id') or result['id']
        extracted_doc_id = doc_id.split('_')[-1] if '_' in str(doc_id) else doc_id

        # Build a prompt using only this document as context so we get one answer per doc
        context = result.get('content', '')
        prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

        try:
            resp = adv_rag.llm.invoke([prompt])
            answer_text = resp.content if hasattr(resp, 'content') else str(resp)
        except Exception as e:
            answer_text = f"LLM error: {e}"

        rows.append({
            "QueryId": str(qid),
            "Rank": int(rnk),
            "DocumentId": str(extracted_doc_id),
            "Similarity": float(result.get('similarity_score', None)) if result.get('similarity_score', None) is not None else None,

        })

# Save top-5 answers per query (one answer per retrieved document)
submission_df = pd.DataFrame(rows)
submission_df.to_csv("../submission_advrag_top5.csv", index=False)
print(f"Saved {len(rows)} rows to ../submission_advrag_top5.csv")
print("\nPreview:")
print(submission_df.head(10))

Retrieving relevant documents for the query: How do I cope with drought and erratic rainfall on my farm?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: How can I adapt my farming to drought and erratic rainfall?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: How does drought and erratic rainfall affect my crops?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (afer filtering)
Retrieving relevant documents for the query: What is the risk of drought and erratic rainfall to farming?
Using top_k=5 and threshold=0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (